In [41]:
try:
    from google.colab import drive
except ModuleNotFoundError:
    print("Local environment detected; Google Drive mount skipped.")
else:
    drive.mount("/content/drive")

Local environment detected; Google Drive mount skipped.


In [42]:
# -*- coding: utf-8 -*-
# EXP027: DQN (normal) with selectable MLE/EAP ability estimation
# Based on EXP020. Changes from EXP020:
#   - EAP prior: N(0,1) -> U(-4, 4)  (log_prior = zeros instead of norm.logpdf)
#   - n_quad: 61 -> 62
#   - Config: prior_mean/prior_std replaced by prior_low/prior_high
import copy
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Tuple, cast

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.optimize import minimize_scalar

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


def find_project_root():
    candidates = []
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
        candidates.extend([script_dir, *script_dir.parents])

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "Grad_Research",
            cwd
            / "Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History",
            Path("/content/Grad_Research"),
            Path(
                "/content/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Colab Notebooks/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
        ]
    )

    for root in candidates:
        if (root / "data").is_dir():
            return root

    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "data").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the project root. "
        "In Colab, place the repository at MyDrive/Grad_Research or /content/Grad_Research."
    )


ROOT = find_project_root()
EXP027_DIR = ROOT / "EXP027"
MODEL_DIR = EXP027_DIR / "models"
RESULTS_DIR = EXP027_DIR / "results"

print(f"Device      : {device}")
print(f"Project root: {ROOT}")
print(f"Model dir   : {MODEL_DIR}")
print(f"Results dir : {RESULTS_DIR}")

Device      : mps
Project root: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History
Model dir   : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP027/models
Results dir : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP027/results


In [43]:
@dataclass
class Config:
    # Network
    # input_size=1: state = [theta_hat]
    input_size: int = 1
    first_hidden: int = 50
    second_hidden: int = 30
    dropout_rate: float = 0.0

    # Training
    test_length: int = 40
    gamma: float = 0.1
    memory_capacity: int = 1000
    epsilon: float = 0.1
    batch_size: int = 128
    q_network_iteration: int = 40
    learning_rate: float = 1e-3
    training_size: int = 1000
    validation_size: int = 200
    validation_interval: int = 50

    # Ability estimation: 'MLE' | 'EAP'
    estimation_method: str = "EAP"

    # EAP quadrature (used only when estimation_method='EAP')
    n_quad: int = 62
    prior_low: float = -4.0
    prior_high: float = 4.0

    # Bank / training theta distribution
    bank_type: str = "uncor"  # 'uncor' | 'cor'
    bank_id: int = 1
    prior: str = "normal"  # training theta: 'normal' | 'uniform'
    n_items: int = 500

    # Evaluation reproducibility (aligned with the MFI notebook)
    seed: int = 20260430

In [44]:
def RESPOND(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    p = (1 - c) / (1 + np.exp(-D * a * (theta - b))) + c
    return (np.random.random(size=p.shape) <= p).astype(int)


def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    return (
        D**2
        * a**2
        * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )


def MLE(item_paras, resp, D=1):
    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]

    def mins_likelihood(x):
        logl = 0
        for i in range(len(resp)):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp[i] * np.log(p) + (1 - resp[i]) * np.log(1 - p)
        return logl

    result = cast(
        Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded")
    )
    return np.array(result.x).reshape(
        1,
    )


def MLE_TEST(item_paras, resp, D=1):
    def mins_likelihood(x):
        logl = 0
        for i in range(resp_i.shape[0]):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp_i[i] * np.log(p) + (1 - resp_i[i]) * np.log(1 - p)
        return logl

    theta = np.zeros(resp.shape[1])
    for i in range(resp.shape[1]):
        resp_i = resp[:, i]
        a = item_paras[:, i, 0]
        b = item_paras[:, i, 1]
        c = item_paras[:, i, 2]
        result = cast(
            Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded")
        )
        theta[i] = result.x
    return np.expand_dims(theta, axis=0)


def EAP_quadrature(
    item_paras: np.ndarray,
    resp: np.ndarray,
    n_quad: int = 62,
    prior_low: float = -4.0,
    prior_high: float = 4.0,
    D: float = 1.0,
) -> Tuple[float, float]:
    """EAP mean and posterior variance via quadrature with uniform prior U(prior_low, prior_high).

    Parameters
    ----------
    item_paras : (n_items, 3) array  [a, b, c]
    resp       : (n_items,)  array  0/1

    Returns
    -------
    (eap_mean, eap_variance)
    """
    theta_grid = np.linspace(prior_low, prior_high, n_quad)
    # Uniform prior: constant log-density over the grid, cancels after normalization
    log_prior = np.zeros(n_quad)

    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]
    # shape: (n_items, n_quad)
    p = c[:, None] + (1 - c[:, None]) / (
        1 + np.exp(-D * a[:, None] * (theta_grid[None, :] - b[:, None]))
    )
    p = np.clip(p, 1e-10, 1 - 1e-10)
    log_lik = np.sum(
        resp[:, None] * np.log(p) + (1 - resp[:, None]) * np.log(1 - p),
        axis=0,
    )  # shape: (n_quad,)

    log_post = log_lik + log_prior
    log_post -= log_post.max()
    post = np.exp(log_post)
    post /= post.sum()

    eap_mean = float(np.sum(theta_grid * post))
    eap_var = float(np.sum((theta_grid - eap_mean) ** 2 * post))
    return eap_mean, eap_var


def get_estimation_method(cfg):
    method = cfg.estimation_method.upper()
    if method not in {"MLE", "EAP"}:
        raise ValueError(
            f"Unsupported estimation_method: {cfg.estimation_method!r}. "
            "Use 'MLE' or 'EAP'."
        )
    return method


def estimate_theta_single(cfg, administered_items, responses, current_theta):
    method = get_estimation_method(cfg)
    if method == "EAP":
        theta_hat, _ = EAP_quadrature(
            administered_items,
            responses,
            n_quad=cfg.n_quad,
            prior_low=cfg.prior_low,
            prior_high=cfg.prior_high,
        )
        return theta_hat

    if len(np.unique(responses)) == 1:
        if responses[-1] == 1:
            return current_theta + (item_bank[:, 1].max() - current_theta) / 2
        return current_theta - (current_theta - item_bank[:, 1].min()) / 2
    return float(MLE(administered_items, responses)[0])


def estimate_theta_batch(cfg, item_ids, responses, current_theta):
    method = get_estimation_method(cfg)
    testing_size = responses.shape[1]
    theta_hat = np.zeros(testing_size)

    if method == "EAP":
        for subject in range(testing_size):
            theta_hat[subject], _ = EAP_quadrature(
                item_bank[item_ids[:, subject]],
                responses[:, subject],
                n_quad=cfg.n_quad,
                prior_low=cfg.prior_low,
                prior_high=cfg.prior_high,
            )
        return theta_hat

    idx_full = np.sum(responses, axis=0) == responses.shape[0]
    idx_zero = np.sum(responses, axis=0) == 0
    idx_norm = ~(idx_full | idx_zero)
    theta_hat[idx_full] = (
        current_theta[idx_full] + (item_bank[:, 1].max() - current_theta[idx_full]) / 2
    )
    theta_hat[idx_zero] = (
        current_theta[idx_zero] - (current_theta[idx_zero] - item_bank[:, 1].min()) / 2
    )
    if np.any(idx_norm):
        theta_hat[idx_norm] = np.squeeze(
            MLE_TEST(
                item_bank[item_ids[:, idx_norm]],
                responses[:, idx_norm],
            )
        )
    return theta_hat


def estimation_label(cfg):
    return "EAP_unif" if get_estimation_method(cfg) == "EAP" else "MLE"


def Apply_Positive_Constraint(model, min_value=0.0):
    for param in model.parameters():
        param.data = torch.clamp(param.data, min=min_value)

In [45]:
class Net(nn.Module):
    def __init__(
        self, input_size, first_hidden, second_hidden, action_space, dropout_rate
    ):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, first_hidden)
        self.fc2 = nn.Linear(first_hidden, second_hidden)
        self.out = nn.Linear(second_hidden, action_space)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(self.fc1(x))
        x = F.relu(x)
        x = self.dropout(self.fc2(x))
        x = F.relu(x)
        return self.out(x)

    def initialize(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)

In [46]:
def Choose_Action(item_id, state, epsilon):
    """state: 1-D array shape (1,) = [theta_hat_EAP]"""
    if np.random.rand() >= epsilon:
        state_t = torch.unsqueeze(torch.FloatTensor(state), 0).to(device)
        item_id_t = torch.from_numpy(item_id).to(device).long()
        action_value = eval_net(state_t)
        action_value[:, item_id_t] = -torch.inf
        action = torch.max(action_value, -1)[1].cpu().numpy()
    else:
        if any(item_id):
            action = (
                np.random.choice(np.delete(np.arange(action_space), item_id))
                .astype("int64")
                .reshape(
                    1,
                )
            )
        else:
            action = (
                np.random.choice(np.arange(action_space))
                .astype("int64")
                .reshape(
                    1,
                )
            )
    return action


def Choose_Action_Test(item_id, state):
    """state: 2-D array shape (1, n_subjects) = [[theta_hats_EAP]]"""
    state_t = torch.FloatTensor(state.swapaxes(0, 1)).to(device)
    action_value = eval_net(state_t).detach().cpu().numpy()
    if item_id.shape[0] > 0:
        action_value[
            np.tile(np.arange(item_id.shape[1])[np.newaxis, :], (item_id.shape[0], 1)),
            item_id,
        ] = -np.inf
    return action_value.argmax(axis=1)

In [47]:
def TRAIN(cfg):
    get_estimation_method(cfg)
    best_valid = None
    best_state = None

    loss_func = nn.MSELoss()
    eval_net.train()
    optimizer = optim.Adam(eval_net.parameters(), lr=cfg.learning_rate)

    # memory layout:
    # [state, action, reward, next_state, terminal, next_available(action_space)]
    memory_width = cfg.input_size * 2 + 3 + action_space
    memory = np.zeros((cfg.memory_capacity, memory_width))
    memory_counter = 0
    learn_step_counter = 0

    if cfg.prior == "normal":
        training_theta = np.random.randn(cfg.training_size)
    elif cfg.prior == "uniform":
        training_theta = np.random.uniform(-3, 3, cfg.training_size)
    else:
        raise ValueError(f"Unsupported prior: {cfg.prior!r}.")

    for j in range(cfg.training_size):
        state = np.array([np.random.rand() - 0.5])
        item_id = np.array([]).astype("int64")
        resp = np.array([]).astype("int64")

        for i in range(cfg.test_length):
            action = Choose_Action(item_id, state, cfg.epsilon)
            item_id = np.concatenate((item_id, action))
            resp = np.concatenate((resp, RESPOND(item_bank[action], training_theta[j])))
            reward = FI(item_bank[action,], training_theta[j])

            theta_hat = estimate_theta_single(cfg, item_bank[item_id], resp, state[-1])
            next_state = np.array([theta_hat])

            terminal = float(i == cfg.test_length - 1)
            next_available = np.ones(action_space, dtype=np.float32)
            next_available[item_id] = 0.0

            memory[memory_counter % cfg.memory_capacity, :] = np.hstack(
                (state, action, reward, next_state, terminal, next_available)
            )
            memory_counter += 1
            state = next_state

            if memory_counter >= cfg.batch_size:
                batch_memory = memory[
                    np.random.choice(
                        min(memory_counter, cfg.memory_capacity), cfg.batch_size
                    ),
                    :,
                ]
                next_state_start = cfg.input_size + 2
                next_state_end = next_state_start + cfg.input_size
                terminal_col = next_state_end
                next_available_start = terminal_col + 1

                batch_state = torch.FloatTensor(batch_memory[:, : cfg.input_size]).to(
                    device
                )
                batch_action = torch.LongTensor(
                    batch_memory[:, cfg.input_size : cfg.input_size + 1].astype(int)
                ).to(device)
                batch_reward = torch.FloatTensor(
                    batch_memory[:, cfg.input_size + 1 : cfg.input_size + 2]
                ).to(device)
                batch_next_state = torch.FloatTensor(
                    batch_memory[:, next_state_start:next_state_end]
                ).to(device)
                batch_terminal = torch.FloatTensor(
                    batch_memory[:, terminal_col : terminal_col + 1]
                ).to(device)
                batch_next_available = torch.BoolTensor(
                    batch_memory[:, next_available_start:].astype(bool)
                ).to(device)

                q_eval = eval_net(batch_state).gather(1, batch_action)
                q_next = target_net(batch_next_state).detach()
                q_next = q_next.masked_fill(~batch_next_available, -torch.inf)
                q_next_max = q_next.max(1)[0].view(cfg.batch_size, 1)
                q_next_max = q_next_max.masked_fill(batch_terminal.bool(), 0.0)
                q_target = batch_reward + cfg.gamma * q_next_max
                loss = loss_func(q_eval, q_target)

                optimizer.zero_grad()
                loss.backward()
                Apply_Positive_Constraint(eval_net)
                optimizer.step()

                learn_step_counter += 1
                if learn_step_counter % cfg.q_network_iteration == 0:
                    target_net.load_state_dict(eval_net.state_dict())

        ### Validation ###
        if (j + 1) % cfg.validation_interval == 0:
            eval_net.eval()
            valid_theta = np.random.choice(training_theta, cfg.validation_size)
            valid_bias = np.zeros((cfg.test_length, cfg.validation_size))

            state = np.expand_dims(np.random.rand(cfg.validation_size) - 0.5, axis=0)
            item_id = np.array([])

            for i in range(cfg.test_length):
                action = Choose_Action_Test(item_id, state)
                if i == 0:
                    item_id = action[np.newaxis, :]
                    resp = RESPOND(item_bank[action,], valid_theta)[np.newaxis, :]
                else:
                    item_id = np.concatenate((item_id, action[np.newaxis, :]))
                    resp = np.concatenate(
                        (resp, RESPOND(item_bank[action,], valid_theta)[np.newaxis, :])
                    )

                theta_0 = estimate_theta_batch(cfg, item_id, resp, state[-1])

                state = theta_0[np.newaxis, :]  # shape: (1, validation_size)
                valid_bias[i] = theta_0 - valid_theta

            step_valid = np.transpose(
                np.vstack(
                    (
                        np.arange(1, cfg.test_length + 1),
                        np.mean(valid_bias, axis=1),
                        np.sqrt(np.mean(valid_bias**2, axis=1)),
                        np.mean(abs(valid_bias), axis=1),
                    )
                )
            )
            print("subject: {}\n\n{}\n".format(j + 1, step_valid))

            result_valid = np.mean(step_valid[6:, 1:], axis=0)
            if best_valid is None:
                best_valid = result_valid
                best_state = copy.deepcopy(eval_net.state_dict())
            elif result_valid[1] < best_valid[1]:
                best_valid = result_valid
                best_state = copy.deepcopy(eval_net.state_dict())

            eval_net.train()

    return best_state

In [48]:
def TEST(cfg, theta_test):
    get_estimation_method(cfg)
    # Reset only the evaluation RNG so training does not change the test responses.
    np.random.seed(cfg.seed)

    with torch.no_grad():
        eval_net.eval()
        testing_size = len(theta_test)

        state = np.expand_dims(np.random.rand(testing_size) - 0.5, axis=0)
        item_id = np.array([])
        dqn_step = np.zeros((1, 4))

        for i in range(cfg.test_length):
            action = Choose_Action_Test(item_id, state)
            if i == 0:
                item_id = action[np.newaxis, :]
                resp = RESPOND(item_bank[action,], theta_test)[np.newaxis, :]
            else:
                item_id = np.concatenate((item_id, action[np.newaxis, :]))
                resp = np.concatenate(
                    (resp, RESPOND(item_bank[action,], theta_test)[np.newaxis, :])
                )

            theta_0 = estimate_theta_batch(cfg, item_id, resp, state[-1])

            if i == 0:
                theta = theta_0[np.newaxis, :]
            else:
                theta = np.concatenate((theta, theta_0[np.newaxis, :]))

            dqn_step = np.vstack(
                [
                    dqn_step,
                    np.array(
                        [
                            i + 1,
                            np.mean(theta_0 - theta_test),
                            np.sqrt(np.mean((theta_0 - theta_test) ** 2)),
                            np.mean(abs(theta_0 - theta_test)),
                        ]
                    ),
                ]
            )
            print(
                "step {:g}, bias {:.3f}, rmse {:.3f}, mae {:.3f}".format(*dqn_step[-1])
            )

            state = theta_0[np.newaxis, :]  # shape: (1, testing_size)

        user_id_col = np.repeat(
            np.arange(1, testing_size + 1), cfg.test_length
        ).reshape(-1, 1)
        step_col = np.tile(np.arange(1, cfg.test_length + 1), testing_size).reshape(
            -1, 1
        )
        item_id_col = (item_id + 1).transpose().reshape(-1, 1)
        resp_col = resp.transpose().reshape(-1, 1)
        theta_est_col = theta.transpose().reshape(-1, 1)
        bias_col = (theta - theta_test).transpose().reshape(-1, 1)

        dqn_data = pd.DataFrame(
            np.hstack(
                [user_id_col, step_col, item_id_col, resp_col, theta_est_col, bias_col]
            ),
            columns=["userID", "step", "itemID", "resp", "theta_est", "bias"],
        )
        dqn_summary = pd.DataFrame(
            dqn_step[1:], columns=["step", "Bias", "RMSE", "MAE"]
        )

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        stem = (
            f"{cfg.bank_type}_{cfg.bank_id}_DQN_{estimation_label(cfg)}"
            f"_gamma_{cfg.gamma}"
        )
        dqn_data.to_csv(RESULTS_DIR / f"records_{stem}.csv", index=False)
        dqn_summary.to_csv(RESULTS_DIR / f"summary_{stem}.csv", index=False)
        print(f"\nSaved to {RESULTS_DIR}")

In [49]:
cfg = Config(
    # Network
    input_size=1,  # state = [theta_hat]
    first_hidden=50,
    second_hidden=30,
    dropout_rate=0.0,
    # Training
    test_length=40,
    gamma=0.9,
    memory_capacity=1000,
    epsilon=0.1,
    batch_size=128,
    q_network_iteration=40,
    learning_rate=1e-3,
    training_size=1000,
    validation_size=200,
    validation_interval=50,
    # Ability estimation: 'MLE' | 'EAP'
    estimation_method="MLE",
    # EAP quadrature (used only for EAP)
    n_quad=62,
    prior_low=-4.0,
    prior_high=4.0,
    # Bank / training theta distribution
    bank_type="uncor",
    bank_id=1,
    prior="normal",
    n_items=500,
    # Evaluation reproducibility (same seed as MFI)
    seed=20260430,
)

bank_dir = {
    "uncor": ROOT / "data" / "uncorrelated_banks",
    "cor": ROOT / "data" / "correlated_banks",
}[cfg.bank_type]

item_bank = np.array(
    pd.read_csv(bank_dir / f"item_bank_{cfg.bank_type}_{cfg.bank_id}.csv")[
        ["a", "b", "c"]
    ]
)[: cfg.n_items]
action_space = item_bank.shape[0]

theta_test = np.array(
    pd.read_csv(ROOT / "data" / "theta_true" / f"theta_true_{cfg.bank_id}.csv")["x"]
)

print(f"item bank  : {item_bank.shape}")
print(f"theta_test : {theta_test.shape}")
print(f"\nConfig:\n{cfg}")

item bank  : (500, 3)
theta_test : (5000,)

Config:
Config(input_size=1, first_hidden=50, second_hidden=30, dropout_rate=0.0, test_length=40, gamma=0.9, memory_capacity=1000, epsilon=0.1, batch_size=128, q_network_iteration=40, learning_rate=0.001, training_size=1000, validation_size=200, validation_interval=50, estimation_method='MLE', n_quad=62, prior_low=-4.0, prior_high=4.0, bank_type='uncor', bank_id=1, prior='normal', n_items=500, seed=20260430)


In [50]:
eval_net = Net(
    cfg.input_size, cfg.first_hidden, cfg.second_hidden, action_space, cfg.dropout_rate
).to(device)
target_net = Net(
    cfg.input_size, cfg.first_hidden, cfg.second_hidden, action_space, cfg.dropout_rate
).to(device)
eval_net.initialize()
target_net.initialize()

best_state = TRAIN(cfg)

assert best_state is not None, (
    "No checkpoint was saved. Increase training_size or lower validation_interval."
)
eval_net.load_state_dict(best_state)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = (
    MODEL_DIR
    / f"dqn_{estimation_label(cfg)}_{cfg.bank_type}_{cfg.bank_id}_gamma_{cfg.gamma}.pt"
)
torch.save(eval_net.state_dict(), model_path)
print(f"Model saved to: {model_path}")

TEST(cfg, theta_test)

subject: 50

[[ 1.00000000e+00 -2.61577621e-01  1.40049182e+00  1.20776418e+00]
 [ 2.00000000e+00 -1.17932322e-01  1.72530338e+00  1.39821629e+00]
 [ 3.00000000e+00  3.48445860e-03  1.49405224e+00  1.23128330e+00]
 [ 4.00000000e+00 -6.63572437e-02  1.59757318e+00  1.27206048e+00]
 [ 5.00000000e+00 -9.39401208e-02  1.59454755e+00  1.25906909e+00]
 [ 6.00000000e+00 -1.62326809e-01  1.57679557e+00  1.21220281e+00]
 [ 7.00000000e+00 -1.08375582e-01  1.38682784e+00  1.02359199e+00]
 [ 8.00000000e+00 -1.08618486e-01  1.33309200e+00  1.00195520e+00]
 [ 9.00000000e+00 -7.04719765e-04  1.19864681e+00  9.22831299e-01]
 [ 1.00000000e+01  1.99748248e-02  1.17178188e+00  8.78566440e-01]
 [ 1.10000000e+01  8.38358919e-02  1.01684577e+00  7.72102360e-01]
 [ 1.20000000e+01  9.38879091e-02  9.25824212e-01  7.11631747e-01]
 [ 1.30000000e+01  1.20368242e-01  8.58509592e-01  6.55498910e-01]
 [ 1.40000000e+01  1.06692423e-01  8.28787496e-01  6.32517447e-01]
 [ 1.50000000e+01  8.00332127e-02  8.34964157e-01